# Name That LLM — TF-IDF + LogisticRegression baseline

Char (1-5) + word (1-2) TF-IDF, LogisticRegression with balanced class weights. CV (GroupKFold by prompt): balanced acc ~0.342 (v1, LB 0.345).

v2: cluster test passages by topic + per-cluster Hungarian assignment (one passage per model per prompt). CV ~0.436.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.sparse import hstack
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

KAGGLE_DIR = '/kaggle/input/competitions/name-that-llm'
DATA_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else '../data'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(train.shape, test.shape)

In [ ]:
char = TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 5), sublinear_tf=True, min_df=2)
word = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True, min_df=2,
                       token_pattern=r'(?u)\b\w+\b|[^\w\s]', lowercase=False)
char.fit(train['text'])
word.fit(train['text'])

def featurize(texts):
    return hstack([char.transform(texts), word.transform(texts)]).tocsr()

clf = LogisticRegression(C=10, max_iter=2000, class_weight='balanced')
clf.fit(featurize(train['text']), train['model'])

In [ ]:
# Each prompt is answered once per model -> cluster test passages by topic,
# then assign labels within each cluster so a model is used at most once (Hungarian).
classes = clf.classes_
proba = clf.predict_proba(featurize(test['text']))

if len(test) > 1:
    topic = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2 if len(test) > 50 else 1)
    T = topic.fit_transform(test['text']).toarray()
    clusters = AgglomerativeClustering(n_clusters=None, distance_threshold=0.90,
                                       metric='cosine', linkage='average').fit_predict(T)
else:
    clusters = np.zeros(len(test), dtype=int)
print('clusters:', len(np.unique(clusters)))

pred = classes[proba.argmax(1)].copy()
for k in np.unique(clusters):
    idx = np.where(clusters == k)[0]
    reps = int(np.ceil(len(idx) / len(classes)))  # oversized cluster: allow repeats
    cost = np.hstack([-np.log(proba[idx] + 1e-9)] * reps)
    rows, cols = linear_sum_assignment(cost)
    pred[idx[rows]] = classes[cols % len(classes)]

In [ ]:
VALID = set(train['model'].unique())
assert len(VALID) == 18

submission = pd.DataFrame({'id': test['id'], 'model': pred})
assert submission['id'].is_unique and submission['model'].isin(VALID).all()
submission.to_parquet('submission.parquet', index=False)
submission.head()